# Esperimento composito 02 — ConvLSTM + GRU dei recettori

Partiamo dalla ConvLSTM Large già efficace e restringiamo **una sola parte** della funzione: le transizioni di AMPA, NMDA e GABA vengono affidate a tre `torch.nn.GRU` standard e locali. Tensione, calcio e dodici gate restano sotto lo stesso backbone ConvLSTM.

Il controllo di capacità mantiene la ConvLSTM monolitica ma allarga soltanto la testa MLP fino ad avere quasi gli stessi parametri del composito.

In [ ]:
from pathlib import Path
import subprocess, sys, tempfile

def project_root():
    candidates = [Path.cwd(), Path.cwd().parent]
    work = Path('/kaggle/working')
    if work.exists(): candidates += [p.parent for p in work.glob('*/pyproject.toml')]
    for candidate in candidates:
        marker = candidate / 'pyproject.toml'
        if marker.exists() and 'hay-single-compartment' in marker.read_text(): return candidate
    destination = Path(tempfile.mkdtemp(prefix='hay_composite_02_', dir='/kaggle/working'))
    subprocess.check_call(['git', 'clone', '--depth', '1', 'https://github.com/Zagred47/LearningSingleCompartiment.git', str(destination)])
    return destination

ROOT = project_root(); SRC = ROOT / 'src'
assert (SRC / 'hay_single_compartment').is_dir(), f'Package source missing: {SRC}'
sys.path.insert(0, str(SRC))
print('Project:', ROOT)

In [ ]:
import h5py, json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from hay_single_compartment import INPUT_NAMES, ONTOLOGY_GROUPS, STATE_NAMES, SimulationConfig, generate_dataset, validate_dataset
from hay_single_compartment.dataset import Normalization
from hay_single_compartment.models import build_model
from hay_single_compartment.training import rollout_batch, train_model

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUTPUT_DIR = Path('/kaggle/working/hay_composite_experiment_02') if Path('/kaggle').exists() else ROOT / 'artifacts' / 'composite_02'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATASET = OUTPUT_DIR / 'single_compartment_composite_v1.h5'
print('Device:', DEVICE, '| output:', OUTPUT_DIR)

## 1. Stesso dataset dell'ablation ontologica

Generazione e training mostrano avanzamento, tempo trascorso ed ETA.

In [ ]:
config = SimulationConfig(duration_ms=500.0, warmup_ms=150.0, seed=27182, train_trajectories=24, validation_trajectories=4, test_trajectories=6)
dataset_report = generate_dataset(DATASET, config, progress=True) if not DATASET.exists() else validate_dataset(DATASET)
dataset_report

## 2. Tre modelli controllati

`conv_large` è il punto di partenza. `conv_capacity_control` aggiunge capacità senza ontologia. `conv_receptor_gru` sostituisce soltanto le tre uscite sinaptiche con GRU locali.

In [ ]:
EXPERIMENTS = {
    'conv_large': dict(architecture='conv_lstm', hidden_dim=128, layers=3, width_multiplier=2, head_dim=None),
    'conv_capacity_control': dict(architecture='conv_lstm', hidden_dim=128, layers=3, width_multiplier=2, head_dim=336),
    'conv_receptor_gru': dict(architecture='conv_lstm_receptor_gru', hidden_dim=128, layers=3, width_multiplier=2, head_dim=None, receptor_hidden_dim=32, receptor_layers=1),
}
for name, settings in EXPERIMENTS.items():
    probe = build_model(settings['architecture'], len(STATE_NAMES) + len(INPUT_NAMES), len(STATE_NAMES), **{k: v for k, v in settings.items() if k != 'architecture'})
    settings['parameters'] = sum(parameter.numel() for parameter in probe.parameters())
parameter_table = pd.DataFrame(EXPERIMENTS).T[['architecture', 'parameters']]
parameter_table

## 3. Efficienza dei dati

Il primo passaggio usa 25% e 100% delle traiettorie. Tutti i modelli condividono split, seed, finestre, optimizer e criterio di arresto.

In [ ]:
DATA_FRACTIONS = (0.25, 1.0)
reports = []
for fraction in DATA_FRACTIONS:
    for model_name, settings in EXPERIMENTS.items():
        run_name = f'{model_name}_data_{int(100 * fraction):03d}'
        print('\n' + '=' * 100)
        print(f'Training {run_name}: {settings["parameters"]:,} parameters, {fraction:.0%} data')
        reports.append(train_model(
            DATASET, OUTPUT_DIR / 'models', settings['architecture'], run_name=run_name,
            train_fraction=fraction, epochs=40, sequence_length=128, stride=32, batch_size=32,
            hidden_dim=settings['hidden_dim'], layers=settings['layers'], width_multiplier=settings['width_multiplier'],
            head_dim=settings.get('head_dim'), receptor_hidden_dim=settings.get('receptor_hidden_dim', 32),
            receptor_layers=settings.get('receptor_layers', 1), learning_rate=6e-4, dropout=0.1,
            patience=8, minimum_epochs=18, device=DEVICE, seed=27182, use_amp=True, verbose=True,
        ))
print('All controlled runs completed.')

In [ ]:
comparison = pd.DataFrame([{
    'run': r['run_name'], 'model': r['run_name'].rsplit('_data_', 1)[0], 'data_fraction': r['train_fraction'],
    'train_trajectories': r['train_trajectories'], 'parameters': r['parameters'], 'epochs': r['epochs_trained'],
    'validation_loss': r['best_validation_loss'], 'test_voltage_rmse_mV': r['test']['voltage_rmse_mv'],
    'test_normalized_rmse': r['test']['mean_normalized_rmse'],
} for r in reports]).sort_values(['data_fraction', 'validation_loss'])
comparison.to_csv(OUTPUT_DIR / 'comparison.csv', index=False)
comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for model_name, frame in comparison.groupby('model'):
    frame = frame.sort_values('train_trajectories')
    axes[0].plot(frame.train_trajectories, frame.test_voltage_rmse_mV, marker='o', label=model_name)
    axes[1].plot(frame.train_trajectories, frame.test_normalized_rmse, marker='o', label=model_name)
axes[0].set(title='Voltage', xlabel='training trajectories', ylabel='test RMSE (mV)')
axes[1].set(title='All states', xlabel='training trajectories', ylabel='mean normalized RMSE')
for axis in axes: axis.grid(alpha=.25); axis.legend()
plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'data_efficiency.png', dpi=160)

## 4. Diagnosi per entità e rollout autoregressivo

Valutiamo solo i modelli addestrati sul 100%, senza usare il test per scegliere checkpoint o epoca.

In [ ]:
full_reports = [r for r in reports if r['train_fraction'] == 1.0]
entity_rows = [{'model': r['run_name'].rsplit('_data_', 1)[0], 'entity': entity, 'normalized_rmse': error} for r in full_reports for entity, error in r['test']['per_group_normalized_rmse'].items()]
entity_errors = pd.DataFrame(entity_rows)
entity_errors.to_csv(OUTPUT_DIR / 'entity_errors.csv', index=False)
entity_errors.pivot(index='entity', columns='model', values='normalized_rmse').sort_values('conv_receptor_gru', ascending=False)

In [ ]:
with h5py.File(DATASET, 'r') as h5:
    truth = h5['test/states'][...]
    future_inputs = h5['test/inputs'][...]
HORIZONS_MS = (50, 100, 200, 500)
rollout_rows, predictions = [], {}
for report in full_reports:
    model_name = report['run_name'].rsplit('_data_', 1)[0]
    checkpoint = torch.load(report['checkpoint'], map_location=DEVICE, weights_only=False)
    model = build_model(checkpoint['architecture'], len(STATE_NAMES) + len(INPUT_NAMES), len(STATE_NAMES), **checkpoint['model_kwargs']).to(DEVICE)
    model.load_state_dict(checkpoint['model_state'])
    normalization = Normalization.from_dict(checkpoint['normalization'])
    print(f'\nRollout {model_name} on all {len(truth)} test trajectories...')
    prediction = rollout_batch(model, truth[:, 0], future_inputs, normalization, DEVICE, progress=True)
    predictions[model_name] = prediction
    for horizon in HORIZONS_MS:
        end = int(horizon / config.dt_ms) + 1; error = prediction[:, :end] - truth[:, :end]
        crossings = ((prediction[:, 1:end, 0] >= config.membrane.spike_threshold_mv) & (prediction[:, :end-1, 0] < config.membrane.spike_threshold_mv)).sum()
        teacher_crossings = ((truth[:, 1:end, 0] >= config.membrane.spike_threshold_mv) & (truth[:, :end-1, 0] < config.membrane.spike_threshold_mv)).sum()
        rollout_rows.append({'model': model_name, 'horizon_ms': horizon, 'voltage_rmse_mV': float(np.sqrt(np.mean(error[..., 0]**2))), 'mean_normalized_rmse': float(np.sqrt(np.mean((error / normalization.state_std)**2, axis=(0,1))).mean()), 'teacher_spikes': int(teacher_crossings), 'predicted_spikes': int(crossings)})
rollout_table = pd.DataFrame(rollout_rows)
rollout_table.to_csv(OUTPUT_DIR / 'rollout_comparison.csv', index=False)
rollout_table

In [ ]:
time_ms = np.arange(truth.shape[1]) * config.dt_ms
fig, axes = plt.subplots(2, 1, figsize=(15, 7), sharex=True)
axes[0].plot(time_ms, truth[0, :, 0], color='black', label='teacher', lw=1.2)
axes[1].plot(time_ms, truth[0, :, 1] * 1e3, color='black', label='teacher', lw=1.2)
for name, prediction in predictions.items():
    axes[0].plot(time_ms, prediction[0, :, 0], label=name, alpha=.8)
    axes[1].plot(time_ms, prediction[0, :, 1] * 1e3, label=name, alpha=.8)
axes[0].set(ylabel='V (mV)', title='Composite experiment rollout'); axes[1].set(xlabel='time (ms)', ylabel='Ca i (uM)')
for axis in axes: axis.grid(alpha=.2); axis.legend()
plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'rollout_example.png', dpi=160)

## Criterio di accettazione

La separabilità dei recettori è supportata se il composito supera sia ConvLSTM Large sia il controllo di capacità, soprattutto su errori AMPA/NMDA/GABA e rollout, senza degradare sistematicamente tensione e gate. Solo allora separeremo un secondo sottosistema.

In [ ]:
from shutil import copytree, make_archive, rmtree
import base64, os
from IPython.display import Javascript, display

parameter_table.to_csv(OUTPUT_DIR / 'parameter_table.csv')
(OUTPUT_DIR / 'experiment_definition.json').write_text(json.dumps({'fractions': DATA_FRACTIONS, 'experiments': EXPERIMENTS, 'seed': config.seed}, indent=2), encoding='utf-8')
include_checkpoints = os.environ.get('HAY_DOWNLOAD_CHECKPOINTS', '0') == '1'
archive_source = OUTPUT_DIR; staging = Path('/kaggle/working/hay_composite_02_download')
if not include_checkpoints:
    if staging.exists(): rmtree(staging)
    copytree(OUTPUT_DIR, staging, ignore=lambda path, names: {name for name in names if name.endswith('.pt')})
    archive_source = staging
zip_base = Path('/kaggle/working/hay_composite_experiment_02_complete')
zip_path = Path(make_archive(str(zip_base), 'zip', root_dir=archive_source.parent, base_dir=archive_source.name))
encoded = base64.b64encode(zip_path.read_bytes()).decode('ascii')
display(Javascript(f"""const binary=atob('{encoded}');const bytes=new Uint8Array(binary.length);for(let i=0;i<binary.length;i++)bytes[i]=binary.charCodeAt(i);const blob=new Blob([bytes],{{type:'application/zip'}});const url=URL.createObjectURL(blob);const a=document.createElement('a');a.href=url;a.download='{zip_path.name}';document.body.appendChild(a);a.click();a.remove();setTimeout(()=>URL.revokeObjectURL(url),60000);"""))
print('Download avviato:', zip_path, f'({zip_path.stat().st_size / 2**20:.1f} MiB)', '| checkpoint inclusi:', include_checkpoints)